In [60]:
import ast
import operator as op
import pandas as pd
import numpy as np
from seuif97 import *
from scipy.interpolate import interp1d, LinearNDInterpolator # импортируем методы интерполяции
from math import sqrt, sin, log 
import re
# Находим все идентификаторы с точками


# Расчёт вспомогательных функций
def calc_Pw(T):
        # Saturation pressure at a given temperature
        return pd.Series([tx2p(T[i],0)/ 0.0980665 for i in T.index],T.index)
    
def calc_Hw(T):
        # Enthalpy of water
        return pd.Series([tx2h(T[i],0) for i in T.index],T.index)/4.186
def calc_Hs(T):
        # Enthalpy of steam at the saturation point
        return pd.Series([tx2h(T[i],1) for i in T.index],T.index)/4.186

def calc_T(P):
        # Steam temperature at the saturation point at a given pressure
        return pd.Series([px2t((P[i]+1)* 0.0980665,1) for i in P.index],P.index)

def calc_H(P,T):
        Hs=pd.Series([pt2h((P[i]+1)* 0.0980665,T[i])  for i in T.index],T.index)
        Hs=Hs/4.186
        Hs_=pd.Series([tx2h(T[i],1) for i in T.index],T.index)
        Hs_=Hs_/4.186
        Hs[Hs_>Hs]=Hs_[Hs_>Hs]
        return Hs

def clip(df,min_,max_):
    return df.clip(min_,max_)


def add_curve(Curves,Name,X,F):
        n=np.shape(X);
        if len(n)==1: # Интерполяция одномерных функций
            Curves.update({Name:interp1d(X,F,bounds_error=False, fill_value='extrapolate')})
        else:         # Интерполяция многомерных функций
            Curves.update({Name:LinearNDInterpolator(X, F,rescale=True)})
        return Curves

def preprocess_dotted_variables(code):    
    pattern = r'\b([a-zA-Z_][a-zA-Z0-9_]*(\.[a-zA-Z_][a-zA-Z0-9_]*)+)\b'
    def replacer(match):
        return match.group(1).replace('.', '___')
    return re.sub(pattern, replacer, code)

def rereplace(Name):
    return Name.replace('___','.')
    


class ExpressionEvaluator:
    def __init__(self, df):
        df.columns=[i.replace('.','___') for i in df.keys() ]
        self.calculated=[]
        self.df = df
        self.ops = {
            ast.Add: op.add,
            ast.Sub: op.sub,
            ast.Mult: op.mul,
            ast.Div: op.truediv,
            ast.Pow: op.pow,
            ast.USub: op.neg,

            # Операции сравнения
            ast.Eq: op.eq, ast.NotEq: op.ne,
            ast.Lt: op.lt, ast.LtE: op.le,
            ast.Gt: op.gt, ast.GtE: op.ge,
            
            # Логические операции
            ast.And: lambda x, y: x & y,  # Для pandas Series
            ast.Or: lambda x, y: x | y,   # Для pandas Series            
        }
        
        self.functions = {
            'fig':self.calc_curve,
            'clip':clip,
            'tw2p':calc_Pw,
            'tw2h':calc_Hw,
            'ts2h':calc_Hs,
            'px2t':calc_T,
            'pt2h':calc_H,
            'sqrt': np.sqrt,
            'sin': np.sin,
            'log': np.log,
            'sum': np.sum,
            # Добавьте другие функции по необходимости
        }
        self.curvs={}
        
    def get_df(self):
        df=self.df.copy()
        df.columns=[i.replace('___','.') for i in df.keys() ]
        return df
        
    def calc_curve(self,Name,*X):
        #print('calc_curve')
        #print('Name:',Name)
        #print('X:',*X)
        return self.curvs[Name](*X)
        
    def add_curve(self,Name,X,F):
        n=np.shape(X);
        if len(n)==1: # Интерполяция одномерных функций
            self.curvs.update({Name:interp1d(X,F,bounds_error=False, fill_value='extrapolate')})
        else:         # Интерполяция многомерных функций
            self.curvs.update({Name:LinearNDInterpolator(X, F,rescale=True)})
        return self.curvs
    

    def eval_expr(self, expr):
        node = ast.parse(expr, mode='eval')
        return self._eval(node.body)

    def _eval(self, node):
        if isinstance(node, ast.Num):  # Число
            return node.n
        elif isinstance(node, ast.Str):  # Строковый литерал
            return node.s    
        elif isinstance(node, ast.Name):  # Столбец DataFrame
            return self.df[node.id]
        elif isinstance(node, ast.BinOp):  # Бинарная операция (+, -, *, /)
            left = self._eval(node.left)
            right = self._eval(node.right)
            return self.ops[type(node.op)](left, right)
        elif isinstance(node, ast.UnaryOp):  # Унарная операция (например, -x)
            return self.ops[type(node.op)](self._eval(node.operand))
        elif isinstance(node, ast.Call):  # Функции (sqrt(), sin() и т.д.)
            func_name = node.func.id
            args = [self._eval(arg) for arg in node.args]
            return self.functions[func_name](*args)
        elif isinstance(node, ast.Compare):  # Обработка сравнений
            left = self._eval(node.left)
            # Обрабатываем цепочку сравнений (например: 30 <= age <= 40)
            result = left
            for operation, comparator in zip(node.ops, node.comparators):
                right = self._eval(comparator)
                result = self.ops[type(operation)](result, right)
            return result        
        else:
            raise ValueError(f"Неподдерживаемая операция: {type(node).__name__}")

    def calc_expr(self,expr,new_column_name):
        # Вычисляем выражение
        try:
            result = self.eval_expr(expr)
            self.calculated.append(new_column_name)
            print(f"Выражение:{new_column_name} = {expr} OK!\n") #Результат: {result}\n
        except Exception as e:
            print(f"Ошибка в выражении '{new_column_name}={expr}': {str(e)}")
            result = df.eval(expression)
        
        # Если результат - Series (один столбец), добавляем в DataFrame
        if isinstance(result, (pd.Series, np.ndarray)):
            
            kwargs = {new_column_name: result}
            #self.df[new_column_name] = #result.values
            self.df = self.df.assign(**kwargs)
        else:
            # Если результат скалярный, применяем ко всем строкам
            self.df[new_column_name] = result
        return self.df    
        
    def calc_expressions(self,expressions):
        # Вычисление выражений
        for expr, col_name in expressions:
            print(col_name,'=',expr)
            self.calc_expr(expr, col_name)
        return  self.df 
        
    def calc_expressions_eq(self,expressions):
        # Вычисление выражений
        for expression in expressions:
            expression=preprocess_dotted_variables(expression)
            col_name, expr = expression.split('=')
            #print(col_name,'=',expr)
            self.calc_expr(expr, col_name)
        return  self.df 
    
    def get_calc(self):
        return self.df[self.calculated]




In [78]:
expressions_eq=['TsKPU=Tkpu',
                'TcKPU=Tr_KPU',
                'Dsp=D2_5',
                'GsuvEG=clip(GsuvEG,0,10000)',
                'GKPU=GsuvEG',
                'Tt=(Tsob1+Tsob2)/2',       # Усреднение температуры
                
                'H0=pt2h(P0,T0)',           # Enthalpy of superheated steam
                'Hsp=pt2h(Psp,Tsp)',        # Enthalpy of industrial extraction
                'Hst=pt2h(Pt,Tt)',          # Enthalpy of heat extraction
                'HcPSG=tw2h(TcPSG)',        # Enthalpy of PSG condensate
                'HsKPU=ts2h(TcKPU)',        # Enthalpy of steam at KPU
                'dT_c=TcPSG-Tr_PSG',        # PSG temperature undercooling
                'HwKPU=tw2h(TsuvEG)',       # Enthalpy of SUV before KPU
                'Hw_KPU=tw2h(Tr_KPU)',      # Enthalpy of SUV after KPU

                'HwPSG=tw2h(TrPSG)',        # Enthalpy of water before PSG
                'Hw_PSG=tw2h(Tr_PSG)',      # Enthalpy of water after PSG

                'Tt_c=tw2p(Pt)',            # Calculation of Pt_c condensation temperature in PSG by pressure Pt
                'PcPSG=tw2p(TcPSG)',     # Saturated steam pressure at PSG condensate temperature
                'Pt_plus_1=Pt+1',           # Pressure in absolute units kgf/cm2
                'P_PSG=tw2p(Tr_PSG)',     # Saturated steam pressure at temperature

                #"D0_=fig('D0',Pt,N)"
               ]

In [79]:
manager = EnhancedInfluxDBManager()
date_start='2025-10-21 05:00'
end_time='2025-10-21 07:00'
raw_data = manager.read_data(measurement="calc2",start_time=date_start,filter_='',end_time=end_time,tags={'temperature':'2'})
raw_data=raw_data[[i for i in raw_data.keys() if 'TA5' in i]]



                    SELECT * FROM calc2 
                    WHERE temperature='2' AND time >= '2025-10-21 05:00:00' AND time <= '2025-10-21 07:00:00'
                AND "name" =~ //  tz('Etc/GMT-3')


In [81]:
raw_data.columns=[i[4:] for i in raw_data.keys()]

In [82]:
evaluator = ExpressionEvaluator(raw_data)
result = evaluator.calc_expressions_eq(expressions_eq)

Выражение:TsKPU = Tkpu OK!

Выражение:TcKPU = Tr_KPU OK!

Выражение:Dsp = D2_5 OK!

Выражение:GsuvEG = clip(GsuvEG,0,10000) OK!

Выражение:GKPU = GsuvEG OK!

Выражение:Tt = (Tsob1+Tsob2)/2 OK!

Выражение:H0 = pt2h(P0,T0) OK!

Выражение:Hsp = pt2h(Psp,Tsp) OK!

Выражение:Hst = pt2h(Pt,Tt) OK!

Выражение:HcPSG = tw2h(TcPSG) OK!

Выражение:HsKPU = ts2h(TcKPU) OK!

Выражение:dT_c = TcPSG-Tr_PSG OK!

Выражение:HwKPU = tw2h(TsuvEG) OK!

Выражение:Hw_KPU = tw2h(Tr_KPU) OK!

Выражение:HwPSG = tw2h(TrPSG) OK!

Выражение:Hw_PSG = tw2h(Tr_PSG) OK!

Выражение:Tt_c = tw2p(Pt) OK!

Выражение:PcPSG = tw2p(TcPSG) OK!

Выражение:Pt_plus_1 = Pt+1 OK!

Выражение:P_PSG = tw2p(Tr_PSG) OK!



In [83]:
evaluator.get_calc()

,TsKPU,TcKPU,Dsp,GsuvEG,GKPU,Tt,H0,Hsp,Hst,HcPSG,HsKPU,dT_c,HwKPU,Hw_KPU,HwPSG,Hw_PSG,Tt_c,PcPSG,Pt_plus_1,P_PSG
2025-10-21 05:00:00+03:00,116.790244,37.0,1.000000,0.0,0.0,92.000000,756.485913,681.353495,636.115025,89.392766,613.514841,18.341463,19.438077,37.029029,49.010039,71.000389,0.006233,0.697947,1.000000,0.332172
2025-10-21 06:00:00+03:00,117.255000,37.0,0.970000,0.0,0.0,92.000000,756.087792,681.910195,636.115025,89.647548,613.514841,18.595000,19.047849,37.029029,49.010039,71.000389,-10.197162,0.704723,0.960000,0.332172
2025-10-21 07:00:00+03:00,119.288095,37.0,0.914286,0.0,0.0,91.883333,756.458294,681.998443,636.069879,89.327176,613.514841,18.276191,19.047849,37.029029,49.010039,71.000389,-10.197162,0.696212,0.885714,0.332172


# Что нужно сделать:

1. Загрузить из базы measurement hour
2. Выполнить расчёт (все котлы, все турбины, бойлерные)
3. Сохранить в measurement calc
4. Сделать в Dash возможность добавлять строки расчёта и подгружмть характеристики из базы, инициировать расчёт для диапазона данных


In [71]:
# Загрузка данных из measured
from InfluxDatabase import *

manager = EnhancedInfluxDBManager()
date_start='2025-10-21 05:00'
end_time='2025-10-22 05:00'
raw_data = manager.read_data(measurement="calc2",start_time=date_start,end_time=end_time,tags={'temperature':'2'},filter_='')
raw_data
#out_=raw_data.iloc[0:1]



                    SELECT * FROM calc2 
                    WHERE temperature='2' AND time >= '2025-10-21 05:00:00' AND time <= '2025-10-22 05:00:00'
                AND "name" =~ //  tz('Etc/GMT-3')


,Boilers.D_B1,Boilers.D_B2,Boilers.D_B3,Boilers.D_B4,Boilers.G7O4f,Boilers.G7O4r,Boilers.GTm1f,Boilers.GTm1f_d,Boilers.GTm1r,Boilers.GTm1r_d,...,TA9.P_B,TA9.T0,TA9.T0L1,TA9.T0L2,TA9.T_L1,TA9.T_L2,TimeShift,fleet,nBoilers,temperature
2025-10-21 05:00:00+03:00,0.0,0.0,0.0,50.353060,337.115385,326.500000,1407.745152,1405.691891,1673.569219,1672.770287,...,-0.2,33.292683,38.219512,83.560976,34.000000,33.000000,454.000,None,None,2
2025-10-21 06:00:00+03:00,0.0,0.0,0.0,52.829852,339.236842,326.210526,1396.974474,1400.309524,1664.929206,1659.121431,...,-0.2,33.900000,38.000000,80.825000,34.000000,33.000000,343.000,None,None,2
2025-10-21 07:00:00+03:00,0.0,0.0,0.0,53.388419,337.961538,326.730769,1397.510381,1399.793008,1652.405757,1654.667407,...,-0.2,34.000000,38.000000,78.571429,34.000000,33.000000,247.000,None,None,2
2025-10-21 08:00:00+03:00,0.0,0.0,0.0,53.547648,336.500000,326.717949,1399.288587,1396.967430,1660.938046,1663.490708,...,-0.2,34.000000,38.000000,75.902439,34.000000,33.000000,88.000,None,None,2
2025-10-21 09:00:00+03:00,0.0,0.0,0.0,53.657661,328.375000,326.425000,1396.970483,1395.618613,1657.737759,1662.255817,...,-0.2,34.000000,38.000000,74.023810,34.000000,33.000000,247.000,None,None,2
2025-10-21 10:00:00+03:00,0.0,0.0,0.0,54.697770,335.162162,325.891892,1391.081071,1393.190230,1656.202699,1655.209750,...,-0.2,34.000000,38.000000,71.850000,34.000000,33.000000,470.000,None,None,2
2025-10-21 11:00:00+03:00,0.0,0.0,0.0,54.923339,341.050000,326.637500,1396.125757,1395.138622,1641.823224,1640.068171,...,-0.2,34.000000,38.000000,69.642857,34.000000,33.000000,161.000,None,None,2
2025-10-21 12:00:00+03:00,0.0,0.0,0.0,54.341822,347.807692,331.000000,1397.275395,1396.154768,1636.923086,1639.930952,...,-0.2,34.378378,38.000000,68.432432,34.729730,33.567568,306.000,None,None,2
2025-10-21 13:00:00+03:00,0.0,0.0,0.0,53.301442,339.216216,332.608108,1401.915400,1397.882935,1646.321071,1648.429265,...,-0.2,35.000000,38.000000,66.538462,35.000000,34.000000,107.000,None,None,2
2025-10-21 14:00:00+03:00,0.0,0.0,0.0,52.226615,343.463415,334.000000,1403.193190,1397.237788,1654.835836,1658.988870,...,-0.2,35.000000,38.090909,64.795455,35.000000,34.000000,204.000,None,None,2


In [72]:
#    out_['TA13.I']=(1-out_['TA13.II'])*((out_['TA13.Tr_PSG1']-out_['TA13.Tr'])>1.7)*1*(out_['TA13.N']>10)*(out_['TA13.Gr']>400)
eq1= 'TA13.I=(1-TA13.II)*((TA13.Tr_PSG1-TA13.Tr)>1.7)*1*(TA13.N>10)*((TA13.Gr)>400)'

In [73]:
#evaluator.df

In [74]:
Data=raw_data.head()
evaluator = ExpressionEvaluator(Data)
result = evaluator.calc_expressions_eq(['TA13.II=((TA13.Tr_PSG2-TA13.Tr_PSG1)>1.7)*1*(TA13.N>10)',eq1])

Выражение:TA13___II = ((TA13___Tr_PSG2-TA13___Tr_PSG1)>1.7)*1*(TA13___N>10) OK!

Выражение:TA13___I = (1-TA13___II)*((TA13___Tr_PSG1-TA13___Tr)>1.7)*1*(TA13___N>10)*((TA13___Gr)>400) OK!



In [76]:
evaluator.get_calc()

,TA13___II,TA13___I
2025-10-21 05:00:00+03:00,0,0
2025-10-21 06:00:00+03:00,0,0
2025-10-21 07:00:00+03:00,0,0
2025-10-21 08:00:00+03:00,0,0
2025-10-21 09:00:00+03:00,0,0


In [ ]:
#    out_['TA13.II']=((out_['TA13.Tr_PSG2']-out_['TA13.Tr_PSG1'])>1.7)*1*(out_['TA13.N']>10)*(out_['TA13.Gd']>400)
#    out_['TA13.I']=(1-out_['TA13.II'])*((out_['TA13.Tr_PSG1']-out_['TA13.Tr'])>1.7)*1*(out_['TA13.N']>10)*(out_['TA13.Gd']>400)
#    out_['TA13.K']=(1-out_['TA13.I']-out_['TA13.II'])*1*(out_['TA13.N']>10)
#    #out_[[]]
#
#    out_['TA12.II']=((out_['TA12.Tr_PSG2']-out_['TA12.Tr_PSG1'])>1.7)*1*(out_['TA12.N']>10)*(out_['TA12.Gd']>400)
#    out_['TA12.I']=(1-out_['TA12.II'])*((out_['TA12.Tr_PSG1']-out_['TA12.Tr'])>1.7)*1*(out_['TA12.N']>10)*(out_['TA12.Gd']>400)
#    out_['TA12.K']=(1-out_['TA12.I']-out_['TA12.II'])*1*(out_['TA12.N']>10)
#    #out_[[]]
#
#    out_['TA11.II']=((out_['TA11.Tr_PSG2']-out_['TA11.Tr_PSG1'])>1.7)*1*(out_['TA11.N']>10)*(out_['TA11.Gd']>400)
#    out_['TA11.I']=(1-out_['TA11.II'])*((out_['TA11.Tr_PSG1']-out_['TA11.Tr'])>1.7)*1*(out_['TA11.N']>10)*(out_['TA11.Gd']>400)
#    out_['TA11.K']=(1-out_['TA11.I']-out_['TA11.II'])*1*(out_['TA11.N']>10)